### SubAGENT developement

A deepagent can delegate the tasks to multiple subagents. (sync subagents ) (async subagents)

- subagent can resolve the context bloating problem.
- subagent isolates detailed work and then main agent receives the final product.(not the dozens of tool calls associated with it. )



 tools, subagents(sync, async), skills(md file), backend(state, FileSystem, in memory), checkpointers(memory), context(AGENTS.md), middlewares , interrupt_on, system_prompt

In [ ]:
import os
from dotenv import load_dotenv
from tavily import TavilyClient
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent
from langsmith.utils import get_env_var



model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google_genai"
)

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def web_search(query: str):
    '''Search the web for information on a query'''
    return tavily_client.search(query=query)

# each subagent is just a dict: name + description (used by the main agent to decide when to delegate) + prompt + tools
research_subagent = {
    "name": "researcher",
    "description": "Used to research a topic on the web and summarize findings.",
    "system_prompt": "You are a research assistant. Use web_search to find information, then give a concise summary.",
    "tools": [web_search],
}

agent = create_deep_agent(
    model=model,
    subagents=[research_subagent],
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Use the researcher subagent to find out who Balen Shah  is and summarize it in 3 bullet points."
    }]
})

result["messages"][-1].content

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is LangSmith?"
        }
    ]
})

print(result["messages"][-1].content)

In [35]:
import os

print(os.getenv("LANGSMITH_PROJECT"))
print(os.getenv("LANGSMITH_ENDPOINT"))
print(os.getenv("LANGSMITH_API_KEY"))

my-deep-project
https://api.smith.langchain.com
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


In [ ]:
from google import genai
from langsmith import wrappers


def main():
    # genai.Client() reads GOOGLE_API_KEY / GEMINI_API_KEY from the environment
    gemini_client = genai.Client()

    # Wrap the Gemini client to enable LangSmith tracing
    client = wrappers.wrap_gemini(
        gemini_client,
        tracing_extra={
            "tags": ["gemini", "python"],
            "metadata": {
                "integration": "google-genai",
            },
        },
    )

    # Make a traced Gemini call
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="Explain quantum computing in simple terms.",
    )

    print(response.text)


if __name__ == "__main__":
    main()

### STREAMING MESSAGES

streaming messages is a process in which the llm sends the text to the client (text by text) as it is generating instead waiting for the entire buffer to wait and increase latency.

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os 
load_dotenv()
llm=ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b"
)

for chunk in llm.stream("Explain what an AI agent is in simple language."):
    print(chunk.content, end="", flush=True)

An **AI agent** is like a tiny, smart helper that lives inside a computer program.  
It can **see** or **hear** information (like a user’s questions or data from sensors), **think** about what to do next, and **act** (reply, move a robot, change a setting, etc.).  

Think of it as a very helpful robot that only works in software:

1. **Senses** – It gets input from the world (text, numbers, images, etc.).  
2. **Decides** – It uses a set of rules or a learned model to choose the best action.  
3. **Acts** – It produces an output or changes something in its environment.

So an AI agent is just a small program that can perceive, decide, and act—much like a virtual assistant, a game character, or a smart home device.